# 04 — Baseline Fine-Tuning IndoBERT

Notebook ini melatih IndoBERT pada split yang sama dengan BiLSTM.

## Apa yang terjadi saat fine-tuning?

1. Teks dipecah menjadi subword oleh tokenizer IndoBERT.
2. Setiap token menjadi vektor 768 dimensi.
3. 12 lapisan Transformer membaca konteks dua arah.
4. Representasi `[CLS]` masuk ke kepala klasifikasi 2 neuron.
5. Softmax menghasilkan probabilitas `safe` dan `unsafe`.
6. Bobot pre-trained disesuaikan (fine-tuned) dengan data kita.

Karena model jauh lebih besar dari BiLSTM, GPU sangat disarankan.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook biasanya dijalankan dari folder notebooks/. Kita pindah ke root
# project agar semua path artefak konsisten.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "artifacts" / "data"
BERT_DIR = PROJECT_ROOT / "artifacts" / "bert" / "input"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "notebook_runs"
for path in (DATA_DIR, BERT_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("BERT input   :", BERT_DIR)
print("Runs dir     :", RUNS_DIR)

In [ ]:
from app.experiment.configs import DataConfig
from app.experiment.data_pipeline import prepare_data

config_path = DATA_DIR / "data_config.json"
if config_path.exists():
    data_config = DataConfig.from_dict(json.loads(config_path.read_text(encoding="utf-8")))
else:
    candidates = [
        PROJECT_ROOT / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT.parent / "old" / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT / "data.csv",
    ]
    csv_path = next((path for path in candidates if path.exists()), None)
    if csv_path is None:
        raise FileNotFoundError(
            "CSV tidak ditemukan. Set DataConfig.csv_input secara manual "
            "ke file dengan kolom: nama_produk;text;label"
        )
    data_config = DataConfig(
        csv_input=str(csv_path),
        output_dir=str(DATA_DIR),
        bert_input_dir=str(BERT_DIR),
        sort_source=True,
    )

# prepare_data bersifat deterministik dan cepat; aman dijalankan ulang.
bundle = prepare_data(data_config)
print(bundle.summary_frame().to_string(index=False))

In [ ]:
from app.experiment.configs import BertConfig

QUICK_RUN = True

config = BertConfig()
if QUICK_RUN:
    config.epochs = 1
    config.max_len = 128
    config.early_stopping_patience = 1

pd.DataFrame([config.to_dict()]).T.rename(columns={0: "nilai"})

## Lihat tokenisasi

Contoh di bawah menunjukkan bagaimana kalimat komposisi dipecah menjadi
subword. Token `[CLS]` dipakai untuk klasifikasi, `[SEP]` menandai akhir
segmen.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
example = bundle.X_val_text[0]
encoded = tokenizer(example, truncation=True, max_length=config.max_len)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print("Teks:", example[:300])
print("Jumlah token:", len(tokens))
print(tokens[:60])

## Fine-tuning dan evaluasi

Holdout hanya diprediksi sekali di akhir. Early stopping dan pemilihan model
memakai validation.

In [ ]:
from app.experiment.bert_pipeline import train_final_bert

OUT_DIR = RUNS_DIR / "bert_baseline"
output = train_final_bert(
    bundle,
    config,
    output_dir=OUT_DIR,
    hpo_info={"mode": "baseline_notebook"},
    verbose=True,
)
print(json.dumps(output["metrics"]["validation"], indent=2))
print(json.dumps(output["metrics"]["holdout"], indent=2))

## Visualisasi hasil

Kurva training BERT dibaca dari `training_log.json`; metrik validation
dihitung setiap epoch. Perhatikan epoch terbaik dan apakah validation loss
mulai naik.

In [ ]:
log = json.loads((OUT_DIR / "training_log.json").read_text(encoding="utf-8"))
eval_rows = [row for row in log if "eval_loss" in row]
train_rows = [row for row in log if "loss" in row and "eval_loss" not in row]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if train_rows:
    axes[0].scatter([row["epoch"] for row in train_rows], [row["loss"] for row in train_rows], label="Train loss")
if eval_rows:
    axes[0].plot([row["epoch"] for row in eval_rows], [row["eval_loss"] for row in eval_rows], marker="o", label="Eval loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

for key, label in (("eval_f1", "F1"), ("eval_roc_auc", "ROC-AUC"), ("eval_average_precision", "AP")):
    points = [(row["epoch"], row[key]) for row in eval_rows if key in row]
    if points:
        axes[1].plot([p[0] for p in points], [p[1] for p in points], marker="o", label=label)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from app.experiment.reporting import load_probabilities, plot_confusion, plot_roc_pr

y_val, p_val, _ = load_probabilities(OUT_DIR / "probs_val.csv")
y_hold, p_hold, _ = load_probabilities(OUT_DIR / "probs_holdout.csv")

plot_confusion(y_hold, p_hold, OUT_DIR / "confusion_holdout.png", title="IndoBERT baseline — frozen holdout")
plot_roc_pr(
    {"Validation": (y_val, p_val), "Frozen holdout": (y_hold, p_hold)},
    OUT_DIR / "roc_pr.png",
    title="IndoBERT baseline",
)
print("Plot tersimpan di", OUT_DIR)

## Catatan GPU dan memori

| Gejala | Solusi |
|---|---|
| CUDA out of memory | Turunkan `batch_size`, `max_len`, atau naikkan `gradient_accumulation_steps` |
| Training sangat lambat | Pastikan `torch.cuda.is_available()` bernilai True |
| Download model gagal | Unduh di login node, set `HF_HUB_OFFLINE=1` |
| Hasil berbeda antar run | Set seed yang sama + `PYTHONHASHSEED` sebelum Python |

Untuk HPO IndoBERT, gunakan notebook 05. Baseline ini menjadi pembanding.